In [1]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

In [2]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = "/kaggle/input/qrt-quant-quest-iit-bombay-2025/"

features = pd.read_parquet( "./features.parquet")

universe = pd.read_parquet("./universe.parquet")
 
returns = pd.read_parquet( "./returns.parquet")

defining the functions of training XG model after building the features 

In [7]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from scipy.stats import spearmanr

# -----------------------------
# Utility functions
# -----------------------------
def cs_rank(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-sectional rank per day scaled to [0,1]."""
    return df.rank(axis=1, pct=True, method="average")

def safe_div(a, b, eps=1e-8):
    return a / (b + eps)

def make_feature_dict(features: pd.DataFrame) -> dict:
    """
    features.parquet expected format:
    columns are MultiIndex: (feature_name, stock_id)
    index is dates
    """
    if not isinstance(features.columns, pd.MultiIndex):
        raise ValueError("Expected MultiIndex columns in features: (feature_name, stock_id)")
    feat_names = features.columns.get_level_values(0).unique()
    out = {}
    for f in feat_names:
        out[f] = features[f].copy()
    return out

def build_engineered_features(feat: dict) -> dict:
    """Add a few robust engineered features (all still DataFrame date x stock)."""
    eng = dict(feat)

    # lagged base features (t-1 usable at time t)
    for k in ["macd", "trix", "relative_strength_index", "chaikin_money_flow",
              "trend_1_3", "trend_5_20", "trend_20_60", "volatility_20", "volatility_60"]:
        if k in feat:
            eng[f"{k}_lag1"] = feat[k].shift(1)
            eng[f"{k}_delta_5"] = feat[k].shift(1) - feat[k].shift(6)

    # interactions
    if "trend_5_20" in feat and "volatility_20" in feat:
        eng["trend5_20_over_vol20"] = safe_div(feat["trend_5_20"].shift(1), feat["volatility_20"].shift(1))
    if "trend_1_3" in feat and "trend_20_60" in feat:
        eng["trend_spread"] = feat["trend_1_3"].shift(1) - feat["trend_20_60"].shift(1)

    # RSI reversal zones
    if "relative_strength_index" in feat:
        rsi = feat["relative_strength_index"].shift(1)
        eng["rsi_oversold"] = (rsi < 30).astype(float)
        eng["rsi_overbought"] = (rsi > 70).astype(float)

    return eng

def preprocess_features_for_day(eng_feats: dict, day, universe_day: pd.Series):
    """
    Build X_day matrix (n_stocks x n_features) for a single day.
    - cross-sectional rank transform each feature on that day
    - keep only tradable universe
    """
    cols = []
    names = []
    tradable = universe_day[universe_day == 1].index

    for name, df in eng_feats.items():
        s = df.loc[day, tradable]
        # rank cross-sectionally
        s = s.rank(pct=True, method="average")
        cols.append(s.values.reshape(-1, 1))
        names.append(name)

    X_day = np.hstack(cols) if len(cols) > 0 else np.empty((len(tradable), 0))
    return tradable, X_day, names

# def build_train_matrix(features: pd.DataFrame, universe: pd.DataFrame, returns: pd.DataFrame,
#                        train_start: str, train_end: str):
#     """
#     Returns:
#       X, y, meta
#     y target = next-day cross-sectional rank of returns
#     """
#     feat = make_feature_dict(features)
#     eng = build_engineered_features(feat)

#     dates = universe.loc[train_start:train_end].index
#     X_list, y_list = [], []
#     meta = []

#     # target uses t+1 return => last day cannot be used
#     for i in range(len(dates) - 1):
#         d = dates[i]
#         d_next = dates[i + 1]

#         u = universe.loc[d]
#         tradable, X_day, feat_names = preprocess_features_for_day(eng, d, u)

#         # target: next day returns rank in cross-section (tradable @ day d)
#         r_next = returns.loc[d_next, tradable]
#         y_day = r_next.rank(pct=True, method="average").values

#         # filter NaN rows
#         mask = np.isfinite(X_day).all(axis=1) & np.isfinite(y_day)
#         X_day = X_day[mask]
#         y_day = y_day[mask]
#         kept_stocks = tradable[mask]

#         if len(y_day) == 0:
#             continue

#         X_list.append(X_day)
#         y_list.append(y_day)
#         meta.extend([(d, s) for s in kept_stocks])

#     X = np.vstack(X_list)
#     y = np.concatenate(y_list)
#     return X, y, feat_names, meta

def build_train_matrix(features: pd.DataFrame, universe: pd.DataFrame, returns: pd.DataFrame,
                            train_start: str, train_end: str):
    """
    Fully vectorized matrix builder. Generates X and y 100x faster by eliminating day-by-day loops.
    """
    print("1. Engineering features...")
    feat = make_feature_dict(features)
    eng = build_engineered_features(feat)
    feat_names = list(eng.keys())

    print("2. Vectorizing target and masking...")
    # Slice universe for the exact dates to save memory
    u_slice = universe.loc[train_start:train_end]
    
    # Calculate the target (y): Rank the returns cross-sectionally, then shift(-1) 
    # to align today's features with TOMORROW'S return rank.
    target_returns = returns.where(universe == 1).rank(axis=1, pct=True, method="average")
    target_slice = target_returns.shift(-1).loc[train_start:train_end]
    
    # Flatten the 2D target matrix into a 1D Numpy array
    y_flat = target_slice.to_numpy().flatten()

    print("3. Vectorizing features...")
    X_cols = []
    for name in feat_names:
        # Mask against universe and rank globally in one sweep
        masked = eng[name].where(universe == 1)
        ranked = masked.rank(axis=1, pct=True, method="average")
        
        # Slice dates, flatten to 1D, and append
        col_flat = ranked.loc[train_start:train_end].to_numpy().flatten()
        X_cols.append(col_flat)

    print("4. Stacking and filtering NaNs...")
    # Combine all 1D feature arrays into a single 2D matrix (Samples x Features)
    X_mat = np.column_stack(X_cols)

    # Find rows where we have both a valid target AND valid features
    valid_mask = np.isfinite(y_flat) & np.isfinite(X_mat).all(axis=1)

    X = X_mat[valid_mask]
    y = y_flat[valid_mask]

    # Optional: Generate Meta (Dates & Stocks) vectorially if you need to track predictions
    # date_idx = np.repeat(u_slice.index.values, u_slice.shape[1])
    # stock_idx = np.tile(u_slice.columns.values, u_slice.shape[0])
    # valid_dates = date_idx[valid_mask]
    # valid_stocks = stock_idx[valid_mask]
    # meta = list(zip(valid_dates, valid_stocks))
    meta = [] # Keeping empty by default to save Kaggle RAM

    print(f"Done! Final shape: X={X.shape}, y={y.shape}")
    return X, y, feat_names, meta

def train_xgb_ranker(X_train, y_train, X_val=None, y_val=None):
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    if X_val is not None and y_val is not None:
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    else:
        model.fit(X_train, y_train, verbose=False)
    return model

def daily_ic(model, features, universe, returns, start_date, end_date):
    feat = make_feature_dict(features)
    eng = build_engineered_features(feat)
    dates = universe.loc[start_date:end_date].index

    ics = []
    for i in range(len(dates) - 1):
        d = dates[i]
        d_next = dates[i + 1]
        u = universe.loc[d]
        tradable, X_day, _ = preprocess_features_for_day(eng, d, u)

        r_next = returns.loc[d_next, tradable]
        mask = np.isfinite(X_day).all(axis=1) & np.isfinite(r_next.values)
        if mask.sum() < 10:
            continue

        preds = model.predict(X_day[mask])
        ic = spearmanr(preds, r_next.values[mask]).correlation
        if np.isfinite(ic):
            ics.append(ic)

    return {
        "mean_ic": float(np.mean(ics)) if len(ics) else np.nan,
        "std_ic": float(np.std(ics)) if len(ics) else np.nan,
        "num_days": int(len(ics))
    }


def scores_to_weights(scores_day: pd.Series, universe_day: pd.Series) -> pd.Series:
    tradable = universe_day[universe_day == 1].index
    w = pd.Series(0.0, index=universe_day.index)

    s = scores_day.loc[tradable].copy()
    s = s.rank(pct=True, method="average")     # optional but robust
    a = s - s.mean()                           # dollar-neutral

    denom = a.abs().sum()
    if denom > 0:
        w.loc[tradable] = a / denom            # gross=1
    return w


def get_weights(hist_features: pd.DataFrame, today_universe: pd.Series) -> dict:
    """
    Generate portfolio weights using the trained XGBoost model on historical features.
    
    Parameters:
    -----------
    hist_features : pd.DataFrame
        Historical feature data up to (but not including) today.
        - Index: Datetime (chronological order)
        - Columns: MultiIndex with levels (feature_name, stock_id)
    
    today_universe : pd.Series
        Tradable stocks on today.
        - Index: Stock identifiers
        - Values: 0 or 1 (1 = tradable)
    
    Returns:
    --------
    dict[str, float]
        Dictionary mapping stock_id (str) to weight (float).
        Ensures dollar neutrality, unit capital, and max weight constraints.
    """
    # Handle empty history case
    if hist_features.shape[0] == 0:
        return {}
    
    # Build engineered features
    feat = make_feature_dict(hist_features)
    eng = build_engineered_features(feat)
    
    # Extract features for the last available day
    last_day = hist_features.index[-1]
    tradable, X_day, _ = preprocess_features_for_day(eng, last_day, today_universe)
    
    # Handle case with no valid features for any stock
    if X_day.shape[0] == 0:
        return {}
    
    # Generate scores using the trained model
    scores = model.predict(X_day)
    scores_series = pd.Series(scores, index=tradable)
    
    # Convert scores to weights using dollar-neutral scaling
    weights = scores_to_weights(scores_series, today_universe)
    
    # Return as dictionary, excluding zero weights
    return weights[weights != 0].to_dict()

Executing the functions

In [4]:
import time

print("Status: Formatting datetime indices...")
t0 = time.time()
# Ensure datetime index
features.index = pd.to_datetime(features.index)
universe.index = pd.to_datetime(universe.index)
returns.index = pd.to_datetime(returns.index)
print(f"[✓] Indices formatted in {time.time() - t0:.2f} seconds.\n")

# Example split
train_start, train_end = "2005-01-03", "2016-12-30"
val_start, val_end = "2017-01-02", "2019-12-31"

print(f"Status: Building Training Data ({train_start} to {train_end})...")
t1 = time.time()
X_train, y_train, feat_names, _ = build_train_matrix(
    features, universe, returns, train_start, train_end
)
print(f"[✓] Training matrix built in {time.time() - t1:.2f} seconds.")
print(f"    Train shape: {X_train.shape}, {y_train.shape}\n")

print(f"Status: Building Validation Data ({val_start} to {val_end})...")
t2 = time.time()
X_val, y_val, _, _ = build_train_matrix(
    features, universe, returns, val_start, val_end
)
print(f"[✓] Validation matrix built in {time.time() - t2:.2f} seconds.")
print(f"    Val shape: {X_val.shape}, {y_val.shape}\n")

print("Status: Training XGBoost Ranker...")
t3 = time.time()
model = train_xgb_ranker(X_train, y_train, X_val, y_val)
print(f"[✓] Model training complete in {time.time() - t3:.2f} seconds.\n")

print("Status: Calculating Daily Information Coefficient (IC)...")
t4 = time.time()
metrics = daily_ic(model, features, universe, returns, val_start, val_end)
print(f"[✓] IC evaluation complete in {time.time() - t4:.2f} seconds.\n")

print("=====================================")
print("Validation IC Stats:")
print("=====================================")
print(metrics)

Status: Formatting datetime indices...
[✓] Indices formatted in 0.09 seconds.

Status: Building Training Data (2005-01-03 to 2016-12-30)...
1. Engineering features...
2. Vectorizing target and masking...
3. Vectorizing features...
4. Stacking and filtering NaNs...
Done! Final shape: X=(2935626, 44), y=(2935626,)
[✓] Training matrix built in 175.00 seconds.
    Train shape: (2935626, 44), (2935626,)

Status: Building Validation Data (2017-01-02 to 2019-12-31)...
1. Engineering features...
2. Vectorizing target and masking...
3. Vectorizing features...
4. Stacking and filtering NaNs...
Done! Final shape: X=(746262, 44), y=(746262,)
[✓] Validation matrix built in 93.52 seconds.
    Val shape: (746262, 44), (746262,)

Status: Training XGBoost Ranker...
[✓] Model training complete in 291.28 seconds.

Status: Calculating Daily Information Coefficient (IC)...
[✓] IC evaluation complete in 155.53 seconds.

Validation IC Stats:
{'mean_ic': 0.0033679662047892767, 'std_ic': 0.10449688722790218, '

In [ ]:
import time

print("Status: Generating portfolio iteratively (this may take a few minutes)...")
start_time = time.time()

# 1. Generate the portfolio using your XGBoost get_weights function
# We use a 1-year validation period to test it quickly before running the full 20-year span.
xgb_portfolio = generate_portfolio(
    get_weights,
    features,
    universe,
    "2010-01-01",
    "2010-12-31",
)

generation_time = time.time() - start_time
print(f"[✓] Portfolio generated in {generation_time:.2f} seconds.")

print("Status: Running backtest...")

# 2. Backtest the generated portfolio using the daily returns
# The backtest_portfolio function takes (portfolio_weights, daily_returns, universe, plot_flag, verbose_flag)
sr, pnl = backtest_portfolio(
    xgb_portfolio.loc["2010"], 
    returns.loc["2010"], 
    universe.loc["2010"], 
    True,   # Set to True to plot the PnL curve
    True    # Set to True to print detailed stats
)

print("=====================================")
print("Backtest Results (2010-01-01 to 2010-12-31):")
print("=====================================")
print(f"Annualized Sharpe Ratio: {sr:.4f}")


Status: Generating portfolio iteratively (this may take a few minutes)...


100%|██████████| 252/252 [07:49<00:00,  1.86s/it]
/Users/sambhavsinghaditya/Downloads/comet/qrt-quant-challenge-2026-iit-delhi/utils.py:301: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  portfolio = portfolio.fillna(0)


[✓] Portfolio generated in 470.17 seconds.
Status: Running backtest...
Gross Sharpe Ratio:  12.539
Net Sharpe Ratio:  11.922
Turnover %:  93.189


Backtest Results (2010-01-01 to 2010-12-31):
Annualized Sharpe Ratio: 11.9220


TypeError: unsupported format string passed to Series.__format__

In [9]:
print(f"Total PnL: {pnl}")

Total PnL: Date
2010-01-04    0.003705
2010-01-05    0.001950
2010-01-06   -0.000896
2010-01-07   -0.001534
2010-01-08    0.000686
                ...   
2010-12-27    0.001245
2010-12-28    0.002628
2010-12-29    0.000544
2010-12-30    0.000402
2010-12-31    0.001126
Length: 252, dtype: float64


In [ ]:
features[macd].head(1000)

macd                                                         \
                1         2         3         4         5         6    7      
Date                                                                          
2005-01-03  0.000000  0.000000       NaN  0.000000  0.000000  0.000000  NaN   
2005-01-04 -0.004732 -0.013272       NaN -0.000149 -0.002068  0.000122  NaN   
2005-01-05 -0.007316 -0.024657       NaN -0.000298 -0.002582  0.000294  NaN   
2005-01-06 -0.012633 -0.029492       NaN -0.000696 -0.003024  0.000432  NaN   
2005-01-07 -0.017653 -0.024735       NaN -0.001319 -0.005041  0.001372  NaN   
...              ...       ...       ...       ...       ...       ...  ...   
2008-12-15 -0.777421 -1.631596 -0.091065  0.114010  0.489307 -0.040947  NaN   
2008-12-16 -0.763746 -1.466414 -0.063423  0.117969  0.564629 -0.034351  NaN   
2008-12-17 -0.754373 -1.303373 -0.031572  0.120436  0.635972 -0.031954  NaN   
2008-12-18 -0.746978 -1.166584  0.001319  0.120007  0.709385 -0.032112  NaN   
2008-12-19 -0.739508 -1.053095  0.034189  0.119805  0.762883 -0.033516  NaN   

                                     ... chande_momentum_oscillator       \
           8         9         10    ...                       2158 2159   
Date                                 ...                                   
2005-01-03  NaN  0.000000  0.000000  ...                        NaN  NaN   
2005-01-04  NaN -0.002817 -0.000112  ...                        NaN  NaN   
2005-01-05  NaN -0.009337  0.002122  ...                        NaN  NaN   
2005-01-06  NaN -0.011318  0.005346  ...                        NaN  NaN   
2005-01-07  NaN -0.013881  0.009659  ...                        NaN  NaN   
...         ...       ...       ...  ...                        ...  ...   
2008-12-15  NaN -0.038288 -0.151438  ...                  29.864258  NaN   
2008-12-16  NaN -0.001793 -0.121301  ...                  35.754513  NaN   
2008-12-17  NaN  0.031900 -0.081769  ...                  31.279267  NaN   
2008-12-18  NaN  0.063735 -0.046053  ...                  16.320520  NaN   
2008-12-19  NaN  0.097534 -0.010985  ...                  32.075504  NaN   

                                                                      
                 2160 2161 2162 2163 2164       2165 2166       2167  
Date                                                                  
2005-01-03        NaN  NaN  NaN  NaN  NaN        NaN  NaN        NaN  
2005-01-04        NaN  NaN  NaN  NaN  NaN        NaN  NaN        NaN  
2005-01-05        NaN  NaN  NaN  NaN  NaN        NaN  NaN        NaN  
2005-01-06        NaN  NaN  NaN  NaN  NaN        NaN  NaN        NaN  
2005-01-07        NaN  NaN  NaN  NaN  NaN        NaN  NaN        NaN  
...               ...  ...  ...  ...  ...        ...  ...        ...  
2008-12-15 -29.107202  NaN  NaN  NaN  NaN  10.187662  NaN   0.000000  
2008-12-16 -18.239004  NaN  NaN  NaN  NaN   8.219175  NaN  19.999966  
2008-12-17 -22.608299  NaN  NaN  NaN  NaN   3.039074  NaN  23.809423  
2008-12-18 -28.626697  NaN  NaN  NaN  NaN   2.189775  NaN  17.647104  
2008-12-19  -6.166494  NaN  NaN  NaN  NaN  17.252933  NaN  26.315827  

[1000 rows x 47674 columns]